# By Learning Transformation ( Data ingestion / Data egression)

# DATA ENGINEER - DATA ANALYST
## ETL DEVELOPER 
## DATA CURATION DEV


Transformation we are going to achive in two ways , using DSL approach and using SQL approach 

##**1. Data Munging** - (Cleanup) Process of transforming and mapping data from Raw form into Tidy(usable) format with the intent of making it more appropriate and valuable for a variety of downstream purposes such for further Transformation/Enrichment, Egress/Outbound, analytics, Datascience/AI application & Reporting

%md
**Passive Data Munging** - Data Discovery/Data Exploration/ EDA (Exploratory Data Analytics) (every layers ingestion/transformation/analytics/consumption) - Performing an (Data Exploration) exploratory data analysis of the raw data to identify the attributes and patterns. <br>

**Active Data Munging**
1. Combining Data + Schema Evolution/Merging/Merging (Structuring)
2. Validation, Cleansing, Scrubbing - Cleansing (removal of unwanted datasets), Scrubbing (convert raw to tidy)
3. De Duplication and Levels of Standardization () of Data to make it in a usable format (Dataengineers/consumers)

## a. passive Data Munging  
### EDA  / DATA Exploration 

### manually understand the Data  - manual EDA 

1.header 

2. delimiter

3. footer 

4. columns and datatypes

5. comments

6. record count 

7. duplicates / nulls / format issues 



# programatically perform EDA on the Source Data 

In [0]:
cust_df=spark.read.csv("/Volumes/izwd37dev/wd37db/rawdatta/BB2/custsmodified",inferSchema=True).toDF("custid","fname","lname","age","profession")

cust_df.show(10)
cust_df.printSchema()

In [0]:
# Column names

print(cust_df.columns)

print(cust_df.dtypes)  # datatyps -> (col,datatype)

print(cust_df.schema)  # get the schema , structure in spark format


In [0]:
custschema=cust_df.schema
cust_sample_df=spark.read.csv("/Volumes/izwd37dev/wd37db/rawdatta/BB2/cust_sample.txt",schema=custschema)
cust_sample_df.show(4)

cust_sample_df.printSchema()

In [0]:
# count in dataframe

print(cust_df.count())

# check whether DF contains duplicate 

# using distinct - unique records (row level )
# if the entire record is duplicate , then we can remove with distinct 
# distinct() - > record level dedulpication 
# dropDuplicates() without any argument - > records level deduplication

# dropDuplicates([custid]) with argument - > column level deduplication

print(cust_df.distinct().count()) # de duplication on record level

print(cust_df.dropDuplicates().count()) # de duplication on record level

# key column custid 
# as per our business logic - custid is unique 
# custid is unique , unique count of custid in a daframe

df2=cust_df.select("custid")
df2.show(2)
df2.printSchema()

# unique custid count

print(cust_df.select("custid").count()) # total custid count
print(cust_df.select("custid").distinct().count()) # find column level unique count --custid unique count

# when we go with cust_df.select("custid").distinct() , it will return unique custid alone 

# case 2 , need all columns with uniqness based on custid 
# remove duplicates based on the custid

cust_df.dropDuplicates(["custid"]).show()
print(cust_df.dropDuplicates(["custid"]).count())  # deduplicated on column level and return the entire datframe 

#cust_df.display()  # just display content of df in rich format

cust_df.describe().show()  #describes the summary - min, max, mean , count, stddev of each column in df
display(cust_df.describe())

display(cust_df.summary()) # more detail on df summary

#Performance Tip:

"""Percentile calculations (25%, 50%, 75%) require sorting or approx-quantile algorithms across your cluster. On massive datasets (billions of rows), summary() can be significantly slower than describe(). If you only need quick counts and bounds, stick with describe() or specify exact metrics with summary("count", "mean", "min", "max")
"""

In [0]:
cust_df.filter("custid is NULL").show()

In [0]:
cust_df.select("custid").distinct().show(5)

cust_df.dropDuplicates(["custid"]).show(5)

In [0]:
# select -> return the new datframe with selected columns 

df3=cust_df.select("custid","fname")
print(df3.count()) #10005

#unique records

df3.distinct().show()
print(df3.distinct().count()) #10004

%md
# scenarios to create Dataframe 
## suppose i have customer data in diffrent files in same dir - read dir 

## suppose i have customer data in diffrent files in same dir with sub dir as well - read main dir with recursive_lookup enable   

## suppose i have customer data and sales data  in diffrent files in same dir , i want to read only sales data - read dir with file pattern  (/data/sales*)


## suppose i have customer data and sales data  in diffrent files in same dir and sub dir  , i want to read only sales data - read main dir with recursivefilelookup and pathGlobfilter="sales*"

## suppose i have sales data in diff directories - list of path or list of files 


# Schema evolution 

## changes in the sceham 

## Day 1 to Day 5 files have cid ,cname ,age  

## Day 5 to day 10 file have cid ,cname ,age  , profession 


# Day 11 - cid , cname , mobile , profession 

## we achived this writing into some columnar parquet / orc file format 
## while reading the entire data we will use with mergeschema option 


## combining Data -> schema Evolution / Structuring 

In [0]:
#not practiced
# parquet 
stud_df=spark.read.parquet("/Volumes/izwd37dev/wd37db/rawdatta/schema_out/",mergeSchema=True)

stud_df.show()

stud_df.printSchema()



## 1. combine data from diffrent files with changes schema 

In [0]:
stud_df= spark.read.csv("/Volumes/izwd37dev/wd37db/rawdatta/student_part",header=True,inferSchema=True)

stud_df.show()

stud_df.printSchema()

"""
%fs head /Volumes/izwd37dev/wd37db/rawdatta/student_part/stud_part1.csv --sid,sname,age
%fs head /Volumes/izwd37dev/wd37db/rawdatta/student_part/stud_part2.csv --sid,sname,year
%fs head /Volumes/izwd37dev/wd37db/rawdatta/student_part/stud_part3.csv --sid,sname,city
%fs head /Volumes/izwd37dev/wd37db/rawdatta/student_part/stud_part4.csv --sid,sname,city
%fs head /Volumes/izwd37dev/wd37db/rawdatta/student_part/stud_part5.csv --sid,sname,city,age
"""

In [0]:
stud_df1=spark.read.csv("/Volumes/izwd37dev/wd37db/rawdatta/student_part/stud_part4.csv",inferSchema=True,header=True)
stud_df2=spark.read.csv("/Volumes/izwd37dev/wd37db/rawdatta/student_part/stud_part3.csv",header=True,inferSchema=True)

stud_df1.show()

stud_df1.printSchema()

stud_df2.show()

stud_df2.printSchema()

#using Union

complete_stud_df=stud_df1.union(stud_df2)
complete_stud_df.show()
complete_stud_df.printSchema()

In [0]:
# union will work when we have dtafrmes with same number of columns and datatype 
stud_df1=spark.read.csv("/Volumes/izwd37dev/wd37db/rawdatta/student_part/stud_part4.csv",header=True,inferSchema=True)

stud_df2=spark.read.csv("/Volumes/izwd37dev/wd37db/rawdatta/student_part/stud_part3.csv",header=True,inferSchema=True)

stud_df3=spark.read.csv("/Volumes/izwd37dev/wd37db/rawdatta/student_part/stud_part2.csv",header=True,inferSchema=True)

stud_df3.show(1)

stud_df3.printSchema()

stud_df1.union(stud_df3).show()

In [0]:
stud_df1=spark.read.csv("/Volumes/izwd37dev/wd37db/rawdatta/student_part/stud_part1.csv",header=True,inferSchema=True)

stud_df5=spark.read.csv("/Volumes/izwd37dev/wd37db/rawdatta/student_part/stud_part5.csv",header=True,inferSchema=True)

# union will work only with same number of columns and datatypes

stud_df1.printSchema()

stud_df5.printSchema()

stud_df1.union(stud_df5).show()


In [0]:
stud_df1=spark.read.csv("/Volumes/izwd37dev/wd37db/rawdatta/student_part/stud_part1.csv",header=True,inferSchema=True)

stud_df5=spark.read.csv("/Volumes/izwd37dev/wd37db/rawdatta/student_part/stud_part5.csv",header=True,inferSchema=True)

stud_df1.show(1)
stud_df5.show(1)

stud_df1.unionByName(stud_df5,allowMissingColumns=True).show()


In [0]:
# SQL union 

# union --> same number of columns and datatypes based on position 

# in sql union --> retrun unique records 

# in spark DS -  union will allow duplicates 


data1=[(100,"raja",25),(101,"mani",35),(102,"patel",45)]
data2=[(500,"alex",32),(502,"anne",25),(503,"rahul",22),(504,"carry",22)]

data3=[(500,"alex",2000),(502,"anne",2025),(503,"rahul",2022),(504,"carry",2002)]

df1=spark.createDataFrame(data1,['id','name','age']) # 3 records
df2=spark.createDataFrame(data2,['id','name','age']) # 4 records
df3=spark.createDataFrame(data3,['id','name','year']) # 4 records

df1.show()
df2.show()

# combine both df into one 

combined_df=df1.union(df2)

combined_df.show()

print(combined_df.count())

print("id,name age with id , name , year")
df3.show()
combined_df1=df1.union(df3)

combined_df1.show()

data4=[(100,25,"raja"),(101,35,"mani")]


# 
print("column in diffrent order")
df4=spark.createDataFrame(data4,['id','age','name'])

df4.show()

# combined_df1=df1.union(df4) -> will throw error as column type is different 
#combined_df1=df1.union(df4)
#combined_df1.show()

# unionByName
#  default , same number of columns and datatypes based on colun name  
# with allow missing column -> works on differnt column as well 

combined_df2=df1.unionByName(df4)

combined_df2.show()

# id ,name ,age  ---> id,name,year--> id,name,age,year

combined_df3=df1.unionByName(df3,allowMissingColumns=True)

combined_df3.show()



# 2. validation , cleansing , scrubbing  

### handle missing value , handling null 

In [0]:
%fs head /Volumes/izwd37dev/wd37db/rawdatta/BB2/custsmodified

In [0]:
# clean up data which ever not matching with schema 

# reject process - 1

#"/Volumes/izwd37dev/wd37db/rawdatta/BB2/custsmodified"
cust_schema="custid int,fname string, lname string, age int, profession string,error_rec string"
cust_df=spark.read.csv("/Volumes/izwd37dev/wd37db/rawdatta/BB2/custsmodified",header=False,schema=cust_schema, mode="PERMISSIVE",columnNameOfCorruptRecord="error_rec")
cust_df.show(10)

cust_df.printSchema()

valid_df=cust_df.filter("error_rec is  null")
print(valid_df.count())
erro_rc_df=cust_df.filter("error_rec is not null")
erro_rc_df.show()


#erro_rc_df.cache()
erro_rc_df.select("error_rec").show()

erro_rc_df.write.mode("overwrite").csv("/Volumes/izwd37dev/wd37db/rawdatta/BB2/error_table")


# take erro_rec column alone into error_table 
# default spark not working properly this case , workaround cahe() and then perform write operation
# cache() will not work in serverless env
# we can try with compute based databricks environment
# erro_rc_df.select("error_rec").write.mode("overwrite").csv("/Volumes/izwd37dev/wd37db/rawdatta/BB2/error_table")

spark.read.csv("/Volumes/izwd37dev/wd37db/rawdatta/BB2/error_table").show(10,False)

In [0]:
# 2 level cleansing / rejection 

# cleaning up / drop the records 

#  null handling - null record removal 

# null - single column or multiple columns may have null , entire rec may have null 

# single null - remove that recod -> delete rec when col is null 
# mulit col null - remove that recod -> delete rec when col is null and col2 is null 

schema_str="custid int,fname string,lname string,age int,profession string"
cust_df=spark.read.csv("/Volumes/izwd37dev/wd37db/rawdatta/BB2/custsmodified",schema=schema_str,mode="PERMISSIVE")


# handle null -> spark DSL  -> na functions 
# na -> not applicable -> null
cust_df.show()

print(cust_df.count())


# na.drop -> 3 arg -> subset , how , threshold
#  subset -> default is all columns -> option as list of columns 
#  how ->default is  any -> options are  ->   all | any 
#               any -> any one column is the subset is null -> remove that record ->  or
#               all -> all columns are null is the subset  -> remove that record -> and



# if all columns are null remove that record 
all_not_nulll_df=cust_df.na.drop(how="all")

print(all_not_nulll_df.count()) 
#10003


# if any one  columns are null remove that record 
not_nulll_df=cust_df.na.drop(how="any")
not_nulll_df.show(2)

print(not_nulll_df.count())  #9911



# cutid is the key colum it should not have null if cutid is  null remove that record 

cust_id_not_null_df=custdf.na.drop(subset=["cust_id"])

